# Week6_ex2 — Single line-to-ground faults on the loaded 37-bus system

Reproduces Problem 10.64 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a bolted single line-to-ground fault at every bus of the 37-bus design case, solved with the three sequence bus impedance matrices (loads included, pre-fault profile = solved power flow) and checked against PowerWorld; the SLG fault at ORANGE69 gives the generator contributions and the share of buses that sag below 0.75 pu.

![PowerWorld one-line](Images/Week6_ex2_result_image.png)

In [1]:
import warnings
import numpy as np
import pandas as pd
from case37 import S_BASE, BUSES, PW_VM, PW_VA, build_case, solve_pw, residual_band, compare, summarize
from fault37 import (ORDER, KV, GEN_SEQ, BRANCH_ZERO, PW_FAULT, PW_ORANGE_V, PW_ORANGE_GEN,
                     PW_ORANGE_IF, slg_results, slg_at, fault_measure)

# 1. FutureWarning from pandas inside pandapower's branch builder; it prints the local install path
warnings.filterwarnings("ignore", category=FutureWarning)

# 2. Generator sequence impedances (pu on S_BASE); every neutral is solidly grounded (Rn = Xn = 0)
seq = pd.DataFrame(GEN_SEQ, columns=["Bus", "ID", "R1", "X1", "R2", "X2", "R0", "X0", "Rn", "Xn"])
seq.insert(2, "Name", [dict((b, n) for b, n, _ in BUSES)[b] for b in seq.Bus])
seq.insert(3, "kV", seq.Bus.map(KV))
seq

,Bus,ID,Name,kV,R1,X1,R2,X2,R0,X0,Rn,Xn
0,14,1,REDBUD69,69.0,0.00163,0.30,0.00163,0.32,0.00163,0.100,0.0,0.0
1,28,1,ELM345,345.0,0.00000,0.10,0.00000,0.11,0.00000,0.035,0.0,0.0
2,28,2,ELM345,345.0,0.00000,0.10,0.00000,0.11,0.00000,0.035,0.0,0.0
3,31,1,SLACK345,345.0,0.00000,0.07,0.00000,0.07,0.00000,0.030,0.0,0.0
4,44,1,PEACH69,69.0,0.00400,0.11,0.00400,0.12,0.00400,0.030,0.0,0.0
5,48,1,CEDAR69,69.0,0.00156,0.25,0.00156,0.27,0.00156,0.100,0.0,0.0
6,50,1,BIRCH69,69.0,0.00350,0.25,0.00350,0.25,0.00350,0.125,0.0,0.0
7,53,1,PEAR138,138.0,0.00132,0.12,0.00132,0.12,0.00132,0.040,0.0,0.0
8,54,1,PEAR69,69.0,0.00112,0.15,0.00112,0.16,0.00112,0.060,0.0,0.0


In [2]:
# 3. Zero-sequence topology: all 14 transformers are "Unknown" configuration and are left out of
#    the zero sequence, so each voltage level is grounded only by the generators connected to it
online = seq[~((seq.Bus == 14) & (seq.ID == "1"))]
levels = pd.DataFrame({"Buses": pd.Series([kv for _, _, kv in BUSES]).value_counts(),
                       "Grounded generators": online.groupby("kV").Name.agg(lambda s: ", ".join(sorted(set(s))))})
levels["Zero-seq branches"] = [sum(not xf and KV[f] == kv for f, t, *_, xf in BRANCH_ZERO) for kv in levels.index]
levels.index.name = "kV"
levels.sort_index(ascending=False)

,Buses,Grounded generators,Zero-seq branches
kV,,,
345.0,4,"ELM345, SLACK345",3
138.0,11,PEAR138,12
69.0,22,"BIRCH69, CEDAR69, PEACH69, PEAR69",28


In [3]:
# 4. Pre-fault state = the saved power flow, solved with frozen controls and Mvar back-off
net = solve_pw(build_case(), numba=False)

# 5. Residual band: how far each compared quantity moves when the pre-fault solution carries
#    PowerWorld's 0.1 MVA mismatch at any bus (worst case of every bus, summed)
band = residual_band(build_case, fault_measure, numba=False)

# 6. The case stores V and angle in single precision (float32 spacing ~1.2e-7 pu, 2e-6 deg);
#    regulated generator buses sit up to 4.2e-7 pu off their setpoint (PEAR69 1.01000042), so the
#    voltage floor is 5e-7 pu
SRC_PF = "PowerWorld saved case (Bus records)"
rows = []
for b, vm, va in zip(ORDER, PW_VM, PW_VA):
    rows.append(("Pre-fault V", f"V {b} [pu]", net.res_bus.vm_pu[b], vm, 5e-7 + band[f"V0 {b}"], SRC_PF))
    rows.append(("Pre-fault angle", f"delta {b} [deg]", net.res_bus.va_degree[b], va,
                 2e-6 + band[f"delta0 {b}"], SRC_PF))
summarize(compare(rows))

,Group,Source,Rows,Max abs diff,Tightest tolerance,Smallest margin,All pass
0,Pre-fault V,PowerWorld saved case (Bus records),37,0.000007,5.000000e-07,8.000001e-08,True
1,Pre-fault angle,PowerWorld saved case (Bus records),37,0.001338,2.000000e-06,2.000000e-06,True


In [4]:
# 7. Fault Definitions at every bus: Fault 1 Thevenin R, X (= Z1) and Fault 2 (SLG) series
#    impedance Z1 + Z2 + Z0 and current; tolerance = half of the printed digit + residual band
pw = pd.DataFrame(PW_FAULT, columns=["bus", "I1", "a1", "R1 [pu]", "X1 [pu]",
                                     "I [pu]", "Angle [deg]", "R_th [pu]", "X_th [pu]"]).set_index("bus")
res = slg_results(net)
SRC = "PowerWorld Fault Definitions (Run Mode)"
groups = [("R1 [pu]", "Z1 R"), ("X1 [pu]", "Z1 X"), ("R_th [pu]", "Z1+Z2+Z0 R"),
          ("X_th [pu]", "Z1+Z2+Z0 X"), ("I [pu]", "SLG current"), ("Angle [deg]", "SLG angle")]
rows = [(g, f"{g} bus {b}", res.at[b, c], pw.at[b, c], 5e-6 + band[f"{c} {b}"], SRC)
        for b in ORDER for c, g in groups]
summarize(compare(rows))

,Group,Source,Rows,Max abs diff,Tightest tolerance,Smallest margin,All pass
0,Z1 R,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000011,0.000008,True
1,Z1 X,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000011,0.000008,True
2,Z1+Z2+Z0 R,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000018,0.000017,True
3,Z1+Z2+Z0 X,PowerWorld Fault Definitions (Run Mode),37,0.000005,0.000017,0.000015,True
4,SLG current,PowerWorld Fault Definitions (Run Mode),37,0.000240,0.000330,0.000327,True
5,SLG angle,PowerWorld Fault Definitions (Run Mode),37,0.001308,0.004917,0.004841,True


In [5]:
# 8. Single Fault at ORANGE69 (bus 16): fault current, phase voltages at all 37 buses and the
#    phase currents of every generator unit. The faulted phase-a angle at bus 16 is 0/0 noise
i_f, v, gens = slg_at(net, 16)
SRC_SF = "PowerWorld Single Fault, SLG at ORANGE69 (Run Mode)"
rows = [("Fault current", "|I_F| [pu]", abs(i_f), PW_ORANGE_IF[0], 5e-4 + band["I_F ORANGE69"], SRC_SF),
        ("Fault current", "I_F angle [deg]", np.angle(i_f, deg=True), PW_ORANGE_IF[1],
         5e-3 + band["I_F angle ORANGE69"], SRC_SF)]
for b, va, vb, vc, aa, ab, ac in PW_ORANGE_V:
    for ph, mag, ang in zip("abc", (va, vb, vc), (aa, ab, ac)):
        rows.append(("Phase voltage", f"V{ph} bus {b} [pu]", abs(v.at[b, ph]), mag,
                     5e-6 + band[f"V{ph} {b}"], SRC_SF))
        if not (b == 16 and ph == "a"):
            rows.append(("Phase voltage angle", f"V{ph} angle bus {b} [deg]", np.angle(v.at[b, ph], deg=True),
                         ang, 5e-3 + band[f"ang{ph} {b}"], SRC_SF))
for b, gid, ia, ib, ic, aa, ab, ac in PW_ORANGE_GEN:
    for ph, k, mag, ang in zip("abc", range(3), (ia, ib, ic), (aa, ab, ac)):
        rows.append(("Generator current", f"I{ph} G{b}-{gid} [pu]", abs(gens[(b, gid)][k]), mag,
                     5e-6 + band[f"I{ph} G{b}-{gid}"], SRC_SF))
        rows.append(("Generator current angle", f"I{ph} angle G{b}-{gid} [deg]",
                     np.angle(gens[(b, gid)][k], deg=True), ang, 5e-3 + band[f"angI{ph} G{b}-{gid}"], SRC_SF))
sf = compare(rows)
summarize(sf)

,Group,Source,Rows,Max abs diff,Tightest tolerance,Smallest margin,All pass
0,Fault current,"PowerWorld Single Fault, SLG at ORANGE69 (Run ...",2,0.003609,0.001689,0.001382,True
1,Phase voltage,"PowerWorld Single Fault, SLG at ORANGE69 (Run ...",111,0.000011,0.000005,0.000005,True
2,Phase voltage angle,"PowerWorld Single Fault, SLG at ORANGE69 (Run ...",110,0.006080,0.008500,0.006578,True
3,Generator current,"PowerWorld Single Fault, SLG at ORANGE69 (Run ...",24,0.000413,0.000311,0.000303,True
4,Generator current angle,"PowerWorld Single Fault, SLG at ORANGE69 (Run ...",24,0.046308,0.031085,0.027506,True


In [6]:
# 9. Problem 10.64 answer: generator contributions to the ORANGE69 SLG fault (kA uses each
#    unit's own bus base current S_BASE / (sqrt(3) kV))
names = dict((b, n) for b, n, _ in BUSES)
contrib = pd.DataFrame([{"Unit": f"{names[b]} {gid}", "kV": KV[b],
                         "Ia [pu]": abs(i[0]), "Ib [pu]": abs(i[1]), "Ic [pu]": abs(i[2]),
                         "Ia [kA]": abs(i[0]) * S_BASE / (np.sqrt(3) * KV[b])}
                        for (b, gid), i in gens.items()]).set_index("Unit")
contrib.sort_values("Ia [pu]", ascending=False).round(4)

,kV,Ia [pu],Ib [pu],Ic [pu],Ia [kA]
Unit,,,,,
PEAR69 1,69.0,5.3498,2.8184,1.2552,4.4764
SLACK345 1,345.0,2.9999,1.6096,3.0277,0.5020
PEACH69 1,69.0,2.7166,0.3360,0.4770,2.2731
PEAR138 1,138.0,2.5252,1.0906,2.2837,1.0565
ELM345 1,345.0,1.6248,1.0884,1.8238,0.2719
ELM345 2,345.0,1.6248,1.0884,1.8238,0.2719
CEDAR69 1,69.0,1.5908,0.6625,0.3288,1.3311
BIRCH69 1,69.0,0.8996,0.2295,0.4840,0.7527


In [7]:
# 10. Buses whose lowest phase voltage falls below 0.75 pu during the ORANGE69 SLG fault
vmag = v.abs()
low = pd.DataFrame({"Name": [names[b] for b in ORDER], "kV": [KV[b] for b in ORDER],
                    "Va [pu]": vmag.a, "Vb [pu]": vmag.b, "Vc [pu]": vmag.c,
                    "Lowest phase [pu]": vmag.min(axis=1)}, index=vmag.index)
sag = low[low["Lowest phase [pu]"] < 0.75].sort_values("Lowest phase [pu]")
assert (sag[["Vb [pu]", "Vc [pu]"]] > 0.75).all().all()
sag = pd.concat([sag, pd.DataFrame({"Name": [f"{len(sag)} of {len(low)} buses"],
                                     "Lowest phase [pu]": [100 * len(sag) / len(low)]}, index=["Share [%]"])])
sag.round(4)

,Name,kV,Va [pu],Vb [pu],Vc [pu],Lowest phase [pu]
16,ORANGE69,69.0,0.0000,1.1588,1.1527,0.0000
27,MAPLE69,69.0,0.0599,1.1576,1.1553,0.0599
12,OAK69,69.0,0.4060,1.1459,1.1752,0.4060
17,BUCKEYE69,69.0,0.4462,1.1338,1.1514,0.4462
19,APPLE69,69.0,0.4734,1.1308,1.1435,0.4734
15,PECAN69,69.0,0.4960,1.0427,1.0316,0.4960
55,LOCUST69,69.0,0.5243,1.0489,1.0379,0.5243
13,PALM69,69.0,0.5298,1.0868,1.0785,0.5298
37,LIME69,69.0,0.5340,1.0723,1.0806,0.5340
18,WALNUT69,69.0,0.5355,1.0740,1.0824,0.5355
